# UTS Big Data & Predictive Analytics Lanjut (ST153) · Notebook **WORKER 1**
Jalankan di PC Worker 1 (dari 2). Notebook ini menghubungkan PC ke Master dengan dua jalur:
1. **Spark**: menyalakan Spark Worker yang mendaftar ke `spark://MASTER_IP:7077`; executor job Soal 2–4 berjalan di sini.
2. **Kafka**: mengirim detak ke `k1-status` dan menjalankan perintah Master dari `k1-kontrol` (consumer group uji decoupling,
   salinan snapshot untuk `sc.textFile`, ping, laporan ledger offset).

**Urutan:** Sel 1 → Sel 2 → Sel 3 → Sel 4 (loop agen, **biarkan berjalan** sampai Master mengirim `BERHENTI`).
Pastikan versi Python dan PySpark sama dengan PC Master.

## Sel 1 · Konfigurasi dan cek koneksi

In [1]:
# ===== SEL 1: KONFIGURASI WORKER (ISI SESUAI KELOMPOK) =====
# !pip install confluent-kafka pyspark psutil      # sekali saja jika belum terpasang
import os, sys, re, io, json, time, uuid, socket, hashlib, threading, subprocess, platform
from pathlib import Path
from collections import Counter, defaultdict

# ---------------- ISI SESUAI KELOMPOK ANDA ----------------
KELOMPOK     = "CCP"                 # sama dengan notebook Master
NOMOR_WORKER = 1           # 1 ... 2 (unik untuk tiap PC)
MASTER_IP    = "192.168.18.154"         # IP PC Master (broker Kafka + Spark Master)
# -----------------------------------------------------------
MULAI_SPARK_WORKER = True           # False bila Spark Worker sudah dinyalakan manual
CORE_SPARK   = 2                    # core yang disumbangkan PC ini ke cluster (sama dengan CORE_PER_WORKER di Master)
MEMORI_SPARK = "2g"                 # memori untuk executor di PC ini (>= MEMORI_EXECUTOR di Master)

BOOTSTRAP     = f"{MASTER_IP}:9092"
TOPIC_KONTROL = f"{KELOMPOK}-kontrol"
TOPIC_STATUS  = f"{KELOMPOK}-status"
SPARK_MASTER_URL = f"spark://{MASTER_IP}:7077"
HOST = f"{socket.gethostname()}-w{NOMOR_WORKER}"
ROOT_DATA = (Path("C:/Users/Public/uts_bigdata") if os.name == "nt" else Path("/tmp/uts_bigdata")) / KELOMPOK
LOG_DIR = ROOT_DATA / f"log_worker{NOMOR_WORKER}"
LOG_DIR.mkdir(parents=True, exist_ok=True)

from confluent_kafka import Producer, Consumer, TopicPartition, KafkaException
try:
    import psutil
except ImportError:
    psutil = None
import pyspark

def ip_menuju(host, port=9092):
    """IP kartu jaringan PC ini yang dipakai menuju Master (bukan 127.0.0.1)."""
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM); s.connect((host, port))
        ip = s.getsockname()[0]; s.close(); return ip
    except OSError:
        return socket.gethostbyname(socket.gethostname())

def port_terbuka(host, port, timeout=1.5):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

def versi_java():
    java = Path(os.environ["JAVA_HOME"]) / "bin" / "java" if os.environ.get("JAVA_HOME") else "java"
    try:
        r = subprocess.run([str(java), "-version"], capture_output=True, text=True, timeout=20)
        teks = (r.stderr or r.stdout).strip().splitlines()
        return next((b for b in teks if "version" in b), teks[0] if teks else "?")
    except Exception as e:
        return f"tidak ditemukan ({e.__class__.__name__})"

IP_SAYA = ip_menuju(MASTER_IP)
JAVA = versi_java()
prod_status = Producer({"bootstrap.servers": BOOTSTRAP, "acks": "all", "linger.ms": 5, "client.id": HOST})

def lapor(jenis, **isi):
    """Kirim laporan ke Master (key = nama Worker -> urutan pesan per Worker terjaga)."""
    pesan = {"jenis": jenis, "worker": HOST, "nomor": NOMOR_WORKER, "ip": IP_SAYA, "waktu": time.time(), **isi}
    prod_status.produce(TOPIC_STATUS, key=HOST, value=json.dumps(pesan, default=str).encode("utf-8"))
    prod_status.poll(0)

print(f"Worker {HOST} | IP {IP_SAYA} | Python {platform.python_version()} | PySpark {pyspark.__version__} | Java: {JAVA}")
print("Kafka broker  ", BOOTSTRAP, "terjangkau:", port_terbuka(BOOTSTRAP.split(":")[0], int(BOOTSTRAP.split(":")[1])))
print("Spark Master  ", SPARK_MASTER_URL, "terjangkau:", port_terbuka(MASTER_IP, 7077), "(boleh belum, ditunggu di Sel 2)")
meta = prod_status.list_topics(timeout=10)
print("Topic kelompok:", sorted(t for t in meta.topics if t.startswith(KELOMPOK)))
lapor("hidup", python=platform.python_version(), pyspark=pyspark.__version__, executable=sys.executable, java=JAVA,
      spark_worker="belum", konsumen="-", agen=False)
_ = prod_status.flush(10)

Worker Mila-w1 | IP 192.168.18.154 | Python 3.14.8 | PySpark 4.1.3 | Java: openjdk version "21.0.12.1" 2026-08-18 LTS
Kafka broker   192.168.18.154:9092 terjangkau: True
Spark Master   spark://192.168.18.154:7077 terjangkau: True (boleh belum, ditunggu di Sel 2)
Topic kelompok: ['CCP-kontrol', 'CCP-status', 'CCP-wiki-raw']


## Sel 2 · Nyalakan Spark Worker dan daftarkan ke Spark Master
Python milik notebook ini dipasang sebagai `PYSPARK_PYTHON` dan diletakkan di depan `PATH`, sehingga executor memakai Python yang sama.

In [2]:
# ===== SEL 2: Spark Worker daemon =====
SPARK_HOME_PIP = next((p for p in [Path(os.environ.get("SPARK_HOME", "")), Path(pyspark.__file__).parent, Path(pyspark.__file__).parents[2], Path("C:/spark"), Path("D:/spark")] if (p / "bin" / "spark-class.cmd").exists()), Path("C:/spark"))
SPARK_CLASS = SPARK_HOME_PIP / "bin" / ("spark-class.cmd" if os.name == "nt" else "spark-class")
LOG_SPARK = LOG_DIR / "spark-worker.log"
WORK_DIR = ROOT_DATA / f"spark-work-w{NOMOR_WORKER}"
WORK_DIR.mkdir(parents=True, exist_ok=True)

def worker_terdaftar():
    """True bila Spark Master melihat Worker ALIVE dari IP ini."""
    try:
        import urllib.request
        op = urllib.request.build_opener(urllib.request.ProxyHandler({}))
        d = json.loads(op.open(f"http://{MASTER_IP}:8080/json/", timeout=5).read().decode())
        return any(w.get("host") == IP_SAYA and w.get("state") == "ALIVE" for w in d.get("workers", []))
    except Exception:
        return False

def mulai_spark_worker(tunggu_master=600):
    global proc_spark
    if globals().get("proc_spark") is not None and proc_spark.poll() is None:
        print("Spark Worker dari notebook ini sudah berjalan."); return True
    t0, t_cetak = time.time(), 0
    while not port_terbuka(MASTER_IP, 7077):
        if time.time() - t0 > tunggu_master:
            print("Spark Master belum hidup. Jalankan Sel 5 di notebook Master, lalu ulangi sel ini."); return False
        if time.time() - t_cetak > 30:
            print(f"Menunggu Spark Master di {SPARK_MASTER_URL} ... {time.time() - t0:.0f} dtk"); t_cetak = time.time()
        time.sleep(3)
    if worker_terdaftar():
        print("Spark Master sudah melihat Worker dari IP ini (mungkin dinyalakan sebelumnya)."); proc_spark = None; return True
    env = dict(os.environ, SPARK_HOME=str(SPARK_HOME_PIP), SPARK_LOCAL_IP=IP_SAYA, PYSPARK_PYTHON=sys.executable,
               PATH=str(Path(sys.executable).parent) + os.pathsep + os.environ.get("PATH", ""))
    cmd = [str(SPARK_CLASS), "org.apache.spark.deploy.worker.Worker", SPARK_MASTER_URL, "--host", IP_SAYA,
           "--cores", str(CORE_SPARK), "--memory", MEMORI_SPARK, "--work-dir", str(WORK_DIR)]
    awal_log = LOG_SPARK.stat().st_size if LOG_SPARK.exists() else 0
    flags = subprocess.CREATE_NEW_PROCESS_GROUP if os.name == "nt" else 0
    proc_spark = subprocess.Popen(cmd, stdout=open(LOG_SPARK, "ab"), stderr=subprocess.STDOUT, env=env,
                                  cwd=str(LOG_DIR), creationflags=flags)
    t0 = time.time()
    while time.time() - t0 < 90:
        time.sleep(2)
        teks = LOG_SPARK.read_bytes()[awal_log:].decode("utf-8", "replace")
        if "Successfully registered with master" in teks:
            print("\nSpark Worker TERDAFTAR ke", SPARK_MASTER_URL, f"({CORE_SPARK} core, {MEMORI_SPARK})"); return True
        if proc_spark.poll() is not None:
            break
    print("\nSpark Worker belum terdaftar. 15 baris terakhir log:", LOG_SPARK)
    print("\n".join(LOG_SPARK.read_text(encoding="utf-8", errors="replace").splitlines()[-15:]))
    print("Perintah manual:", " ".join(f'"{c}"' if " " in c else c for c in cmd))
    return False

def status_spark_worker():
    if not MULAI_SPARK_WORKER:
        return "manual"
    p = globals().get("proc_spark")
    if p is not None and p.poll() is not None:
        return "MATI"
    return "BERJALAN" if (p is not None or worker_terdaftar()) else "belum"

def hentikan_spark_worker():
    p = globals().get("proc_spark")
    if p is None or p.poll() is not None:
        return
    if os.name == "nt":
        subprocess.run(["taskkill", "/PID", str(p.pid), "/T", "/F"], capture_output=True)
    else:
        p.terminate()
    print("Spark Worker dihentikan.")

proc_spark = globals().get("proc_spark")
if MULAI_SPARK_WORKER:
    mulai_spark_worker()
lapor("hidup", python=platform.python_version(), pyspark=pyspark.__version__, executable=sys.executable, java=JAVA,
      spark_worker=status_spark_worker(), konsumen="-", agen=False)
_ = prod_status.flush(10)

Spark Master sudah melihat Worker dari IP ini (mungkin dinyalakan sebelumnya).


## Sel 3 · Kemampuan Worker: consumer group (Soal 1c) dan penyalin snapshot (Soal 2)

In [3]:
# ===== SEL 3: Consumer uji decoupling + penyalin snapshot =====
class KonsumenUji:
    """Anggota consumer group. Commit manual SETELAH pesan diproses (at-least-once). Mencatat ledger offset per partisi."""
    def __init__(self, cmd, ledger_bersama):
        self.group, self.topic, self.sesi = cmd["group"], cmd["topic"], cmd.get("sesi", 0)
        self.ledger = defaultdict(list)                # partisi -> [[awal, akhir], ...] (urutan pemrosesan)
        ledger_bersama[(self.group, self.sesi)] = self.ledger
        self.belum_commit, self.offset_pertama, self.offset_terakhir = {}, {}, {}
        self.stat = Counter()
        self.kunci_terlihat = KUNCI_TERLIHAT[self.group]       # disimpan lintas sesi: pemrosesan idempoten per Worker
        self.crash, self.tahan_commit = False, False
        self.milik = set()                             # partisi yang sedang di-assign ke Worker ini
        self._henti, self._jeda, self.dijeda = threading.Event(), threading.Event(), False
        self.t = threading.Thread(target=self._jalan, daemon=True, name=f"konsumen-{self.sesi}")
        self.t.start()

    def _commit(self, c, hanya=None):
        offs = [TopicPartition(self.topic, p, o) for p, o in self.belum_commit.items() if hanya is None or p in hanya]
        if not offs:
            return True
        try:
            c.commit(offsets=offs, asynchronous=False)
            for tp in offs:
                self.belum_commit.pop(tp.partition, None)
            self.stat["commit"] += 1
            return True
        except KafkaException as e:                    # mis. REBALANCE_IN_PROGRESS: offset tetap di belum_commit, dicoba lagi
            self.stat["commit_gagal"] += 1
            lapor("galat", sesi=self.sesi, galat=f"commit: {e}")
            return False

    def _on_assign(self, c, parts):
        self.milik |= {p.partition for p in parts}
        if self._jeda.is_set() and parts:
            c.pause(parts)
        com = c.committed(parts, timeout=10) if parts else []
        lapor("assign", sesi=self.sesi, group=self.group, partisi=[p.partition for p in parts],
              committed={tp.partition: tp.offset for tp in com})

    def _on_revoke(self, c, parts):
        dicabut = {p.partition for p in parts}
        if not (self.crash or self.tahan_commit):
            self._commit(c, dicabut)                                  # commit sebelum partisi pindah -> tidak ada duplikat
        self.milik -= dicabut
        for p in dicabut:
            self.belum_commit.pop(p, None)
        lapor("revoke", sesi=self.sesi, partisi=sorted(dicabut))

    def _catat(self, p, o):
        rentang = self.ledger[p]
        if rentang and rentang[-1][1] == o - 1:
            rentang[-1][1] = o
        else:
            rentang.append([o, o])

    def _olah(self, m):
        try:
            d = json.loads(m.value())
        except ValueError:
            self.stat["rusak"] += 1; return
        k = f"{d.get('wiki')}:{d['id']}" if d.get("id") is not None else f"{d.get('wiki')}:uuid:{d.get('meta_id')}"
        if k in self.kunci_terlihat:
            self.stat["ganda_kunci"] += 1                           # sudah pernah diproses -> tidak dihitung lagi
        else:
            self.kunci_terlihat.add(k); self.stat["unik"] += 1

    def _jalan(self):
        c = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": self.group, "auto.offset.reset": "earliest",
                      "enable.auto.commit": False, "session.timeout.ms": 15000, "client.id": HOST})
        c.subscribe([self.topic], on_assign=self._on_assign, on_revoke=self._on_revoke)
        lapor("konsumen_mulai", sesi=self.sesi, group=self.group)
        t_commit = t_lapor = time.time()
        while not self._henti.is_set():
            pesan = c.consume(500, timeout=0.5)
            if self._jeda.is_set():                    # fase 1 penghentian: berhenti memproses lalu commit selagi group stabil
                if not self.dijeda:
                    try:
                        c.pause(c.assignment())
                    except KafkaException:
                        pass
                    ok = False
                    for _ in range(6):
                        if self._commit(c):
                            ok = True; break
                        c.consume(10, timeout=0.5)     # layani callback rebalance, lalu coba commit lagi
                    self.dijeda = True
                    lapor("konsumen_dijeda", sesi=self.sesi, commit_ok=ok, belum_commit=len(self.belum_commit),
                          offset_terakhir=self.offset_terakhir)
                continue                               # pesan yang terlanjur diambil tidak diproses (dibaca ulang sesi berikutnya)
            for m in pesan:
                if m.error() or m.partition() not in self.milik:   # sisa fetch partisi yang sudah dicabut: dilewati
                    continue
                p, o = m.partition(), m.offset()
                self._catat(p, o); self._olah(m)
                self.offset_pertama.setdefault(p, o); self.offset_terakhir[p] = o
                self.belum_commit[p] = o + 1; self.stat["diproses"] += 1
            if not self.tahan_commit and time.time() - t_commit > 2:
                self._commit(c); t_commit = time.time()
            if time.time() - t_lapor > 5:
                lapor("progres", sesi=self.sesi, diproses=self.stat["diproses"], partisi=sorted(self.offset_terakhir),
                      offset_terakhir=self.offset_terakhir); t_lapor = time.time()
        if not (self.crash or self.tahan_commit):
            self._commit(c)
        c.close()
        lapor("konsumen_selesai", sesi=self.sesi, group=self.group, crash=self.crash, **self.stat,
              offset_pertama=self.offset_pertama, offset_terakhir=self.offset_terakhir)

    def jeda(self):
        self._jeda.set()

    def berhenti(self, crash=False):
        self.crash = crash; self._henti.set(); self.t.join(60)

    def aktif(self):
        return self.t.is_alive()

def salin_snapshot(cmd):
    """Tulis pesan topic [awal, akhir) ke path YANG SAMA dengan Master (partisi naik, offset naik) -> MD5 harus sama."""
    path_tujuan = Path(cmd["path"])      # path HARUS sama persis dengan Master agar sc.textFile menemukan berkas
    awal, akhir = {int(k): v for k, v in cmd["awal"].items()}, {int(k): v for k, v in cmd["akhir"].items()}
    path_tujuan.parent.mkdir(parents=True, exist_ok=True)
    tmp = path_tujuan.with_name(path_tujuan.name + ".tmp")
    md5, n, t0 = hashlib.md5(), 0, time.time()
    c = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": f"{KELOMPOK}-salin-{uuid.uuid4().hex[:6]}",
                  "enable.auto.commit": False, "auto.offset.reset": "earliest"})
    with open(tmp, "wb") as f:
        for p in sorted(akhir):
            a, b = awal[p], akhir[p]
            if b <= a:
                continue
            c.assign([TopicPartition(cmd["topic"], p, a)])
            pos, t_diam = a, time.time()
            while pos < b and time.time() - t_diam < 30:
                for m in c.consume(1000, timeout=1.0):
                    if m.error() or m.partition() != p or m.offset() < pos:
                        continue
                    if m.offset() >= b:
                        pos = b; break
                    baris = m.value() + b"\n"; f.write(baris); md5.update(baris)
                    n += 1; pos = m.offset() + 1; t_diam = time.time()
            c.unassign()
    c.close()
    os.replace(tmp, path_tujuan)
    return {"baris": n, "md5": md5.hexdigest(), "detik": round(time.time() - t0, 1), "mb": round(path_tujuan.stat().st_size / 1e6, 1),
            "path": str(path_tujuan)}

def md5_berkas(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for blok in iter(lambda: f.read(1 << 20), b""):
            h.update(blok)
    return {"baris": sum(1 for _ in open(path, "rb")), "md5": h.hexdigest(), "mb": round(Path(path).stat().st_size / 1e6, 1)}

KUNCI_TERLIHAT = defaultdict(set)
LEDGER = {}
print("Kemampuan Worker siap: KonsumenUji, salin_snapshot, md5_berkas.")

Kemampuan Worker siap: KonsumenUji, salin_snapshot, md5_berkas.


## Sel 4 · Loop agen: detak ke Master + menjalankan perintah Master
**Biarkan sel ini berjalan.** Sel selesai sendiri ketika Master mengirim `BERHENTI` (atau Interrupt manual).

In [ ]:
# ===== SEL 4: Loop agen Worker =====
ktl = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": f"{KELOMPOK}-ktl-{HOST}-{uuid.uuid4().hex[:6]}",
                "enable.auto.commit": False})
t0 = time.time()
while True:                                            # topic kontrol dibuat oleh Sel 3 notebook Master
    t_k = ktl.list_topics(TOPIC_KONTROL, timeout=10).topics.get(TOPIC_KONTROL)
    if t_k is not None and t_k.error is None and t_k.partitions:
        break
    if time.time() - t0 > 900:
        raise RuntimeError(f"Topic {TOPIC_KONTROL} belum ada. Jalankan Sel 3 di notebook Master.")
    print(f"Menunggu topic {TOPIC_KONTROL} dibuat Master ..."); time.sleep(5)
# assign langsung di offset akhir: setiap perintah yang dikirim setelah titik ini pasti terbaca (tanpa race consumer group)
ktl.assign([TopicPartition(TOPIC_KONTROL, p, ktl.get_watermark_offsets(TopicPartition(TOPIC_KONTROL, p), timeout=10)[1])
            for p in sorted(t_k.partitions)])
siap = threading.Event(); siap.set()
konsumen = None

def untuk_saya(cmd):
    t = cmd.get("target", "semua")
    return t in ("semua", None) or NOMOR_WORKER in (t if isinstance(t, list) else [t])

def detak():
    lapor("hidup", python=platform.python_version(), pyspark=pyspark.__version__, executable=sys.executable, java=JAVA,
          spark_worker=status_spark_worker(), konsumen=(f"sesi {konsumen.sesi} aktif" if konsumen and konsumen.aktif() else "-"),
          cpu=psutil.cpu_percent() if psutil else None, ram=psutil.virtual_memory().percent if psutil else None, agen=True)

def jalankan_latar(fungsi, jenis, cmd):
    def kerja():
        try:
            lapor(jenis, cmd_id=cmd["cmd_id"], **fungsi(cmd))
        except Exception as e:
            lapor(jenis, cmd_id=cmd["cmd_id"], galat=f"{e.__class__.__name__}: {e}")
        prod_status.flush(10)
    threading.Thread(target=kerja, daemon=True).start()

t_detak, t_cetak, berhenti, n_cmd = 0, time.time(), False, Counter()
print(f"Worker {HOST} menunggu perintah dari {TOPIC_KONTROL} ... (biarkan berjalan)")
try:
    while not berhenti:
        if siap.is_set() and time.time() - t_detak > 5:
            detak(); t_detak = time.time()
        prod_status.poll(0)
        m = ktl.poll(0.5)
        if m is None or m.error():
            if time.time() - t_cetak > 60:
                print(f"{time.strftime('%H:%M:%S')} hidup | perintah diterima: {dict(n_cmd)} | Spark Worker: {status_spark_worker()}")
                t_cetak = time.time()
            continue
        try:
            cmd = json.loads(m.value())
        except ValueError:
            continue
        if not untuk_saya(cmd):
            continue
        aksi = cmd.get("aksi"); n_cmd[aksi] += 1
        print(f"{time.strftime('%H:%M:%S')} perintah {aksi} {({k: v for k, v in cmd.items() if k in ('sesi', 'group', 'crash')})}")
        lapor("ack", cmd_id=cmd["cmd_id"], aksi=aksi)
        if aksi == "PING":
            lapor("pong", cmd_id=cmd["cmd_id"], t_master=cmd["waktu"])
        elif aksi == "KONSUMEN_MULAI":
            if konsumen and konsumen.aktif():
                konsumen.berhenti()
            konsumen = KonsumenUji(cmd, LEDGER)
        elif aksi == "KONSUMEN_TAHAN_COMMIT" and konsumen:
            konsumen.tahan_commit = True
        elif aksi == "KONSUMEN_JEDA" and konsumen:
            konsumen.jeda()
        elif aksi == "KONSUMEN_STOP" and konsumen:
            konsumen.berhenti(crash=bool(cmd.get("crash")))
        elif aksi == "LAPORAN_LEDGER":
            lapor("ledger", cmd_id=cmd["cmd_id"], ledger={str(s): {str(p): r for p, r in lg.items()}
                                                           for (g, s), lg in LEDGER.items() if g == cmd.get("group")})
        elif aksi == "SNAPSHOT":
            jalankan_latar(salin_snapshot, "snapshot", cmd)
        elif aksi == "CEK_FILE":
            jalankan_latar(lambda c: md5_berkas(c["path"]), "snapshot", cmd)
        elif aksi == "BERHENTI":
            berhenti = True
            if konsumen and konsumen.aktif():
                konsumen.berhenti()
            if cmd.get("hentikan_spark"):
                hentikan_spark_worker()
        prod_status.flush(5)
except KeyboardInterrupt:
    print("\nLoop agen dihentikan manual.")
finally:
    lapor("hidup", python=platform.python_version(), pyspark=pyspark.__version__, executable=sys.executable,
          spark_worker=status_spark_worker(), konsumen="berhenti", agen=False)
    _ = prod_status.flush(10)
    ktl.close()
print("Worker selesai. Perintah yang dijalankan:", dict(n_cmd))

Worker Mila-w1 menunggu perintah dari CCP-kontrol ... (biarkan berjalan)
18:32:32 perintah PING {}
18:33:12 hidup | perintah diterima: {'PING': 1} | Spark Worker: BERJALAN
18:34:13 hidup | perintah diterima: {'PING': 1} | Spark Worker: BERJALAN
18:35:13 hidup | perintah diterima: {'PING': 1} | Spark Worker: BERJALAN
18:36:14 hidup | perintah diterima: {'PING': 1} | Spark Worker: BERJALAN
18:37:14 hidup | perintah diterima: {'PING': 1} | Spark Worker: BERJALAN
18:38:14 hidup | perintah diterima: {'PING': 1} | Spark Worker: BERJALAN
18:39:14 hidup | perintah diterima: {'PING': 1} | Spark Worker: BERJALAN
18:40:15 hidup | perintah diterima: {'PING': 1} | Spark Worker: BERJALAN
18:41:15 hidup | perintah diterima: {'PING': 1} | Spark Worker: BERJALAN
18:42:15 hidup | perintah diterima: {'PING': 1} | Spark Worker: BERJALAN
18:43:15 hidup | perintah diterima: {'PING': 1} | Spark Worker: BERJALAN
18:44:16 hidup | perintah diterima: {'PING': 1} | Spark Worker: BERJALAN
18:45:14 perintah KONSUME

In [ ]:
#PING.

## Sel 5 · (Opsional) Matikan Spark Worker secara manual